<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/04_testing_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q --upgrade "torchao>=0.16.0"

In [1]:
import json
import re
import string
from collections import Counter
from pathlib import Path

import torch
from google.colab import drive
from PIL import Image
from peft import PeftModel
from transformers import AutoProcessor, LlavaForConditionalGeneration

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset"
)
DATASET_DIR = DRIVE_ROOT / "llava_dataset"
IMAGE_DIR = DRIVE_ROOT / "images"
ADAPTER_DIR = DRIVE_ROOT / "hf_llava_checkpoints" / "final_adapter"
OUTPUT_DIR = DRIVE_ROOT / "hf_llava_evaluation"

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
EVALUATION_SPLITS = ("test",)

MAX_RECORDS = None
MAX_NEW_TOKENS = 64

def normalize_text(text):
    text = str(text or "").lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def exact_match(prediction, reference):
    return float(normalize_text(prediction) == normalize_text(reference))


def relaxed_match(prediction, reference):
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)
    if not prediction or not reference:
        return 0.0
    return float(
        prediction == reference
        or prediction in reference
        or reference in prediction
    )


def token_f1(prediction, reference):
    predicted_tokens = normalize_text(prediction).split()
    reference_tokens = normalize_text(reference).split()
    if not predicted_tokens or not reference_tokens:
        return 0.0

    common = sum(
        (Counter(predicted_tokens) & Counter(reference_tokens)).values()
    )
    if common == 0:
        return 0.0

    precision = common / len(predicted_tokens)
    recall = common / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def load_records(split):
    path = DATASET_DIR / f"llava_{split}.json"
    if not path.exists():
        raise FileNotFoundError(f"Evaluation file not found: {path}")

    with path.open("r", encoding="utf-8") as file:
        records = json.load(file)

    if MAX_RECORDS is not None:
        records = records[:MAX_RECORDS]
    return records


def get_image_path(record):
    stored_image = Path(str(record["image"]))
    configured_image = IMAGE_DIR / stored_image.name

    if configured_image.exists():
        return configured_image
    if stored_image.is_absolute() and stored_image.exists():
        return stored_image
    raise FileNotFoundError(f"Image not found: {configured_image}")


def generate_answer(model, processor, image, question, device):
    question = str(question).replace("<image>", "").strip()
    prompt = f"USER: <image>\n{question} ASSISTANT:"

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    )

    moved_inputs = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            value = value.to(device)
            if key == "pixel_values":
                value = value.to(dtype=torch.bfloat16)
        moved_inputs[key] = value

    with torch.inference_mode():
        output = model.generate(
            **moved_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )

    prompt_length = moved_inputs["input_ids"].shape[1]
    answer_tokens = output[:, prompt_length:]
    answer = processor.batch_decode(
        answer_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]
    return " ".join(answer.split()).strip()


def evaluate_split(model, processor, split, device):
    records = load_records(split)
    predictions = []
    metric_values = {"exact_match": [], "relaxed_match": [], "token_f1": []}

    for number, record in enumerate(records, 1):
        question = record["conversations"][0]["value"]
        reference = record["conversations"][1]["value"]

        try:
            image_path = get_image_path(record)
            with Image.open(image_path) as image:
                prediction = generate_answer(
                    model, processor, image.convert("RGB"), question, device
                )
            error = None
        except Exception as exception:
            prediction = ""
            error = str(exception)

        metrics = {
            "exact_match": exact_match(prediction, reference),
            "relaxed_match": relaxed_match(prediction, reference),
            "token_f1": token_f1(prediction, reference),
        }
        for name, value in metrics.items():
            metric_values[name].append(value)

        predictions.append({
            "id": record.get("id", f"{split}_{number}"),
            "image": record.get("image", ""),
            "question": question,
            "reference_answer": reference,
            "predicted_answer": prediction,
            "metrics": metrics,
            "error": error,
        })

        if number % 25 == 0 or number == len(records):
            current = {
                name: sum(values) / len(values)
                for name, values in metric_values.items()
            }
            print(
                f"{split.upper()} {number:,}/{len(records):,} | "
                f"EM: {current['exact_match']:.4f} | "
                f"Relaxed: {current['relaxed_match']:.4f} | "
                f"F1: {current['token_f1']:.4f}"
            )

    metrics = {
        name: sum(values) / len(values) if values else 0.0
        for name, values in metric_values.items()
    }
    metrics["records"] = len(records)
    return metrics, predictions


if not torch.cuda.is_available():
    raise RuntimeError("An NVIDIA GPU is required for evaluation.")

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"LoRA adapter not found: {ADAPTER_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda")

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

base_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
).to(device)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("GPU:", torch.cuda.get_device_name(0))
print("Base model:", MODEL_ID)
print("Adapter:", ADAPTER_DIR)

all_metrics = {}
for split in EVALUATION_SPLITS:
    metrics, predictions = evaluate_split(model, processor, split, device)
    all_metrics[split] = metrics

    with (OUTPUT_DIR / f"predictions_{split}.json").open("w", encoding="utf-8") as file:
        json.dump(predictions, file, ensure_ascii=False, indent=2)

with (OUTPUT_DIR / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(all_metrics, file, ensure_ascii=False, indent=2)

print("\nEvaluation complete:")
print(json.dumps(all_metrics, indent=2))
print(f"Predictions and metrics saved to: {OUTPUT_DIR}")


Mounted at /content/drive


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

GPU: NVIDIA A100-SXM4-80GB
Base model: llava-hf/llava-1.5-7b-hf
Adapter: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/hf_llava_checkpoints/final_adapter


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


TEST 25/356 | EM: 0.0400 | Relaxed: 0.3200 | F1: 0.2336
TEST 50/356 | EM: 0.0600 | Relaxed: 0.3000 | F1: 0.3106
TEST 75/356 | EM: 0.1067 | Relaxed: 0.2933 | F1: 0.3573
TEST 100/356 | EM: 0.1400 | Relaxed: 0.3000 | F1: 0.3998
TEST 125/356 | EM: 0.1280 | Relaxed: 0.2800 | F1: 0.3910
TEST 150/356 | EM: 0.1333 | Relaxed: 0.2867 | F1: 0.3927
TEST 175/356 | EM: 0.1257 | Relaxed: 0.2686 | F1: 0.3865
TEST 200/356 | EM: 0.1250 | Relaxed: 0.2600 | F1: 0.3871
TEST 225/356 | EM: 0.1289 | Relaxed: 0.2667 | F1: 0.3893
TEST 250/356 | EM: 0.1400 | Relaxed: 0.3000 | F1: 0.3981
TEST 275/356 | EM: 0.1418 | Relaxed: 0.2945 | F1: 0.3965
TEST 300/356 | EM: 0.1500 | Relaxed: 0.2967 | F1: 0.3984
TEST 325/356 | EM: 0.1631 | Relaxed: 0.3138 | F1: 0.4084
TEST 350/356 | EM: 0.1743 | Relaxed: 0.3143 | F1: 0.4178
TEST 356/356 | EM: 0.1742 | Relaxed: 0.3118 | F1: 0.4175

Evaluation complete:
{
  "test": {
    "exact_match": 0.17415730337078653,
    "relaxed_match": 0.31179775280898875,
    "token_f1": 0.417540959674

In [2]:
!ls -la

total 20
drwxr-xr-x 1 root root 4096 Sep 13 11:43 .
drwxr-xr-x 1 root root 4096 Sep 13 11:26 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwx------ 5 root root 4096 Sep 13 11:43 drive
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [3]:
!pip install -q sacrebleu rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 16.1 MB/s eta 0:00:00


In [4]:
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu

In [6]:
EVALUATION_DIR = DRIVE_ROOT / "hf_llava_evaluation"
PREDICTIONS_FILE = EVALUATION_DIR / "predictions_test.json"
OUTPUT_FILE = EVALUATION_DIR / "metrics_with_bleu_rouge.json"


def normalize(text):
    text = str(text or "").lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


if not PREDICTIONS_FILE.exists():
    raise FileNotFoundError(f"Predictions file not found: {PREDICTIONS_FILE}")

with PREDICTIONS_FILE.open("r", encoding="utf-8") as file:
    predictions = json.load(file)

references = [normalize(item["reference_answer"]) for item in predictions]
hypotheses = [normalize(item["predicted_answer"]) for item in predictions]

bleu = corpus_bleu(hypotheses, [references]).score / 100.0

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_l_values = [
    scorer.score(reference, hypothesis)["rougeL"].fmeasure
    for reference, hypothesis in zip(references, hypotheses)
]
rouge_l = sum(rouge_l_values) / len(rouge_l_values) if rouge_l_values else 0.0

with (EVALUATION_DIR / "metrics.json").open("r", encoding="utf-8") as file:
    existing_metrics = json.load(file)

metrics = dict(existing_metrics.get("test", {}))
metrics.update({
    "bleu": bleu,
    "rouge_l": rouge_l,
    "records": len(predictions),
})

result = {"test": metrics}
with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print("Test metrics including BLEU and ROUGE-L:")
print(json.dumps(result, indent=2))
print(f"Saved to: {OUTPUT_FILE}")



Test metrics including BLEU and ROUGE-L:
{
  "test": {
    "exact_match": 0.17415730337078653,
    "relaxed_match": 0.31179775280898875,
    "token_f1": 0.41754095967435056,
    "records": 356,
    "bleu": 0.07072314667591344,
    "rouge_l": 0.41208594890704636
  }
}
Saved to: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/hf_llava_evaluation/metrics_with_bleu_rouge.json
